# 06 Initial EDA — NBA Lineup Stint Features

## Goal

We want to understand whether the full-season lineup stint dataset is usable for EDA, baseline recommendations, and first-pass modeling.

The final product goal is:

> Given game context, recommend a lineup or substitution option based on historical lineup performance.

Example contexts:
- Need points
- Need defense
- Protecting a lead
- Trailing late in the 4th
- Starter resting
- Opponent has a specific lineup on the floor

This notebook focuses on the first stage: understanding the data and building simple lineup-level summaries.

## 1. Imports and project paths

This notebook uses the team-perspective dataset:

`team_lineup_stint_features_v1_2024_25.parquet`

Each row represents **one team lineup during one stint**. Since each original stint has a home and away team, the team-perspective table has two rows per original stint.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# If the notebook is opened from /notebooks, use the parent as the project root.
# If it is opened from the project root, use the current folder.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_PATH = PROJECT_ROOT / "data/processed/final/team_lineup_stint_features_v1_2024_25.parquet"
MODELING_PATH = PROJECT_ROOT / "data/processed/final/modeling_lineup_stints_v1_2024_25.parquet"
LINEUP_SUMMARY_PATH = PROJECT_ROOT / "data/processed/final/lineup_summary_v1_2024_25.parquet"

print("Project root:", PROJECT_ROOT)
print("Feature path exists:", FEATURE_PATH.exists())

## 2. Load the dataset

Main checks:
- Number of rows and columns
- Number of games
- Number of unique lineups
- Whether the table has the expected team-perspective format

In [ ]:
df = pd.read_parquet(FEATURE_PATH)

print("Shape:", df.shape)
print("Unique games:", df["game_id"].nunique())
print("Unique lineups:", df["lineup_key"].nunique())

display(df.head())

### Takeaway

Expected full-season shape:
- 1,230 games
- One row per team per lineup stint
- The row count should be roughly double the original home/away stint table

This dataset is the main file for EDA and first-pass modeling.

## 3. Dataset health checks

Before modeling, we need to check:
- Missing values
- Zero-duration rows
- Stint duration distribution
- Period distribution
- Extreme target values

Zero-duration rows are usually created by same-clock substitutions. They should generally be removed before modeling because no actual game time passed.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing.head(20))

print("Zero-duration distribution:")
display(df["is_zero_duration"].value_counts(dropna=False))
display(df["is_zero_duration"].value_counts(normalize=True, dropna=False))

In [ ]:
df_nonzero = df[df["duration_seconds"] > 0].copy()
df_30 = df[df["duration_seconds"] >= 30].copy()

print("Original:", df.shape)
print("Nonzero duration:", df_nonzero.shape)
print("At least 30 seconds:", df_30.shape)

print("\nUnique games:")
print("Original:", df["game_id"].nunique())
print("Nonzero:", df_nonzero["game_id"].nunique())
print("30+ seconds:", df_30["game_id"].nunique())

print("\nUnique lineups:")
print("Original:", df["lineup_key"].nunique())
print("Nonzero:", df_nonzero["lineup_key"].nunique())
print("30+ seconds:", df_30["lineup_key"].nunique())

### Takeaway

Use `df_nonzero` when you only need to remove impossible/no-time rows.

Use `df_30` when analyzing rates like `net_points_per_48`, because very short stints can create extreme and noisy per-48 values.

## 4. Stint duration and period distribution

This helps us understand the unit of analysis. A stint is usually short because substitutions happen often.

If most stints are short, individual stint-level outcomes will be noisy. That means lineup-level aggregation is more reliable than ranking single stints.

In [ ]:
display(df_30["duration_seconds"].describe())

plt.figure()
df_30["duration_seconds"].hist(bins=50)
plt.title("Distribution of stint durations (30+ second stints)")
plt.xlabel("Duration seconds")
plt.ylabel("Count")
plt.show()

In [ ]:
period_counts = df_30["period"].value_counts().sort_index()
display(period_counts)

plt.figure()
period_counts.plot(kind="bar")
plt.title("Team-stint rows by period")
plt.xlabel("Period")
plt.ylabel("Rows")
plt.show()

### Takeaway

Most stints are short, often around 1–2 minutes. Overtime periods should appear as period 5 or higher.

This matters because short segments make raw per-48 statistics noisy. We should aggregate by lineup and use minimum-minute thresholds.

## 5. Stint-level scoring distributions

At the individual stint level, `net_points_per_48` can be very extreme because short stints amplify small scoring changes.

This section confirms why raw stint-level performance is not enough for recommendation.

In [ ]:
display(df_30["net_points"].describe())
display(df_30["net_points_per_48"].describe())

plt.figure()
df_30["net_points"].hist(bins=50)
plt.title("Net points per stint")
plt.xlabel("Net points")
plt.ylabel("Count")
plt.show()

plt.figure()
df_30["net_points_per_48"].clip(-250, 250).hist(bins=50)
plt.title("Net points per 48, clipped to [-250, 250]")
plt.xlabel("Net points per 48")
plt.ylabel("Count")
plt.show()

### Takeaway

The mean of `net_points` and `net_points_per_48` should be close to 0 because every stint appears from both teams' perspectives.

Extreme values are expected, especially for short stints. For recommendation, we should aggregate lineups and apply sample-size controls.

## 6. Build lineup-level summary table

This is the most important EDA table.

Each row becomes one team-lineup combination across the season, with:
- Total minutes
- Number of stints
- Number of games
- Points for/against
- Net points
- Points for per 48
- Points against per 48
- Net points per 48

This gives us the first baseline for lineup rankings.

In [ ]:
def build_lineup_summary(data: pd.DataFrame) -> pd.DataFrame:
    summary = (
        data
        .groupby(["team_id", "lineup_key"])
        .agg(
            total_seconds=("duration_seconds", "sum"),
            total_minutes=("duration_minutes", "sum"),
            stints=("stint_id", "count"),
            games=("game_id", "nunique"),
            points_for=("points_for", "sum"),
            points_against=("points_against", "sum"),
            net_points=("net_points", "sum"),
        )
        .reset_index()
    )

    summary["points_for_per_48"] = (
        summary["points_for"] / summary["total_minutes"] * 48
    )

    summary["points_against_per_48"] = (
        summary["points_against"] / summary["total_minutes"] * 48
    )

    summary["net_points_per_48"] = (
        summary["net_points"] / summary["total_minutes"] * 48
    )

    return summary


lineup_summary = build_lineup_summary(df_30)

print("Lineup summary shape:", lineup_summary.shape)
display(lineup_summary.sort_values("total_minutes", ascending=False).head(20))

### Takeaway

The most-used lineups are usually starting lineups or main rotation groups.

For recommendations, total minutes matter because a lineup with 25 minutes of history is much less reliable than a lineup with 300+ minutes.

## 7. Minutes threshold sensitivity

This checks how many lineups remain if we require more historical playing time.

This is important because a lineup recommender should not trust tiny samples too much.

In [ ]:
threshold_rows = []

for minutes in [20, 50, 100, 200, 500]:
    threshold_rows.append({
        "min_minutes": minutes,
        "qualified_lineups": int((lineup_summary["total_minutes"] >= minutes).sum())
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)

plt.figure()
plt.plot(threshold_df["min_minutes"], threshold_df["qualified_lineups"], marker="o")
plt.title("Lineups remaining by minimum-minute threshold")
plt.xlabel("Minimum minutes")
plt.ylabel("Qualified lineups")
plt.show()

### Takeaway

As the minutes threshold increases, the number of usable lineups drops quickly. This suggests:
- Small-sample lineups are common
- We should use shrinkage-adjusted ratings
- Adding previous seasons could help if the team wants more historical sample size

## 8. Best offensive, defensive, and balanced lineups

This section creates three simple recommendation modes:

- Need points: rank by `points_for_per_48`
- Need stops: rank by lowest `points_against_per_48`
- Balanced: rank by `net_points_per_48`

This is a useful baseline before machine learning.

In [ ]:
MIN_MINUTES = 50

qualified = lineup_summary[lineup_summary["total_minutes"] >= MIN_MINUTES].copy()

best_offense = qualified.sort_values("points_for_per_48", ascending=False)
best_defense = qualified.sort_values("points_against_per_48", ascending=True)
best_balanced = qualified.sort_values("net_points_per_48", ascending=False)

print("Qualified lineups:", qualified.shape[0])

print("\nBest offensive lineups:")
display(best_offense.head(20))

print("\nBest defensive lineups:")
display(best_defense.head(20))

print("\nBest balanced lineups:")
display(best_balanced.head(20))

### Takeaway

These rankings are useful for finding interesting lineups, but they are not final recommendations yet.

Reasons:
- They do not fully adjust for opponent strength
- They do not account for score/time context
- They are still affected by sample size
- They are score-per-time, not possession-based ratings

Still, this is a strong first baseline.

## 9. Shrinkage-adjusted lineup ratings

Raw lineup ratings can overrate low-minute lineups.

Shrinkage pulls a lineup's rating toward the team's average. This makes the ranking more stable.

Conceptually:

`adjusted rating = lineup rating weighted by minutes + team average weighted by prior strength`

In [ ]:
team_summary = (
    df_30
    .groupby("team_id")
    .agg(
        team_minutes=("duration_minutes", "sum"),
        team_points_for=("points_for", "sum"),
        team_points_against=("points_against", "sum"),
        team_net_points=("net_points", "sum"),
    )
    .reset_index()
)

team_summary["team_points_for_per_48"] = (
    team_summary["team_points_for"] / team_summary["team_minutes"] * 48
)

team_summary["team_points_against_per_48"] = (
    team_summary["team_points_against"] / team_summary["team_minutes"] * 48
)

team_summary["team_net_points_per_48"] = (
    team_summary["team_net_points"] / team_summary["team_minutes"] * 48
)

lineup_summary = lineup_summary.merge(team_summary, on="team_id", how="left")

# k = prior minutes. Higher k means more conservative shrinkage.
k = 50

lineup_summary["shrunk_points_for_per_48"] = (
    (lineup_summary["total_minutes"] / (lineup_summary["total_minutes"] + k))
    * lineup_summary["points_for_per_48"]
    +
    (k / (lineup_summary["total_minutes"] + k))
    * lineup_summary["team_points_for_per_48"]
)

lineup_summary["shrunk_points_against_per_48"] = (
    (lineup_summary["total_minutes"] / (lineup_summary["total_minutes"] + k))
    * lineup_summary["points_against_per_48"]
    +
    (k / (lineup_summary["total_minutes"] + k))
    * lineup_summary["team_points_against_per_48"]
)

lineup_summary["shrunk_net_points_per_48"] = (
    (lineup_summary["total_minutes"] / (lineup_summary["total_minutes"] + k))
    * lineup_summary["net_points_per_48"]
    +
    (k / (lineup_summary["total_minutes"] + k))
    * lineup_summary["team_net_points_per_48"]
)

qualified_shrunk = lineup_summary[lineup_summary["total_minutes"] >= MIN_MINUTES].copy()

display(
    qualified_shrunk.sort_values("shrunk_points_for_per_48", ascending=False)
    .head(20)
)

display(
    qualified_shrunk.sort_values("shrunk_net_points_per_48", ascending=False)
    .head(20)
)

### Takeaway

Shrinkage-adjusted ratings are better for recommendation than raw per-48 ratings because they reduce the impact of small samples.

For a first baseline recommender:
- Need points → `shrunk_points_for_per_48`
- Need defense → low `shrunk_points_against_per_48`
- Balanced → `shrunk_net_points_per_48`

## 10. Context EDA: score margin and period

A recommendation system should consider game context.

This section checks whether lineup performance differs by:
- Leading/trailing
- Close game vs blowout
- Period
- Home vs away

This is the bridge between simple lineup rankings and context-aware modeling.

In [ ]:
context_df = df_30.copy()

context_df["score_margin_bucket"] = pd.cut(
    context_df["score_margin_start"],
    bins=[-100, -10, -5, 5, 10, 100],
    labels=["trailing_10_plus", "trailing_6_to_10", "close", "leading_6_to_10", "leading_10_plus"],
)

context_summary = (
    context_df
    .groupby(["score_margin_bucket"], observed=False)
    .agg(
        rows=("stint_id", "count"),
        avg_net_points=("net_points", "mean"),
        avg_net_points_per_48=("net_points_per_48", "mean"),
        avg_points_for=("points_for", "mean"),
        avg_points_against=("points_against", "mean"),
    )
    .reset_index()
)

display(context_summary)

period_summary = (
    context_df
    .groupby("period")
    .agg(
        rows=("stint_id", "count"),
        avg_net_points=("net_points", "mean"),
        avg_net_points_per_48=("net_points_per_48", "mean"),
        avg_points_for=("points_for", "mean"),
        avg_points_against=("points_against", "mean"),
    )
    .reset_index()
)

display(period_summary)

### Takeaway

Context features like score margin and period should be included in modeling.

For example:
- A lineup that is good overall may not be the best choice when trailing
- A defensive lineup may be preferred when protecting a lead
- Late-game and overtime situations may need separate analysis

## 11. Player-level on-court summaries

This section expands each lineup into player rows.

Important caution:

This is **not isolated player impact**. It tells us how teams performed while a player was on the floor, but it does not fully control for teammates, opponents, or game context.

In [ ]:
def parse_lineup_key(lineup_key: str) -> list[int]:
    return [int(x) for x in str(lineup_key).split("-") if x != ""]


player_df = df_30.copy()
player_df["player_id"] = player_df["lineup_key"].apply(parse_lineup_key)
player_df = player_df.explode("player_id")
player_df["player_id"] = player_df["player_id"].astype(int)

player_summary = (
    player_df
    .groupby(["team_id", "player_id"])
    .agg(
        total_minutes=("duration_minutes", "sum"),
        net_points=("net_points", "sum"),
        points_for=("points_for", "sum"),
        points_against=("points_against", "sum"),
        stints=("stint_id", "count"),
        games=("game_id", "nunique"),
    )
    .reset_index()
)

player_summary["net_points_per_48"] = (
    player_summary["net_points"] / player_summary["total_minutes"] * 48
)

player_summary["points_for_per_48"] = (
    player_summary["points_for"] / player_summary["total_minutes"] * 48
)

player_summary["points_against_per_48"] = (
    player_summary["points_against"] / player_summary["total_minutes"] * 48
)

display(
    player_summary[player_summary["total_minutes"] >= 100]
    .sort_values("net_points_per_48", ascending=False)
    .head(20)
)

### Takeaway

Player-level summaries help identify players associated with strong or weak lineups.

But these should not be treated as causal. For better individual impact estimates, we would need regression models that control for teammates, opponents, and context.

## 12. Save modeling-ready datasets

This saves two useful handoff files:

1. `modeling_lineup_stints_v1_2024_25.parquet`
   - Stint-level team perspective
   - Removes rows under 30 seconds

2. `lineup_summary_v1_2024_25.parquet`
   - Aggregated lineup-level table
   - Includes raw and shrinkage-adjusted ratings

These are the files teammates can use for EDA and first-pass modeling.

In [ ]:
model_df = df_30.copy()

model_df.to_parquet(MODELING_PATH, index=False)

lineup_summary.to_parquet(LINEUP_SUMMARY_PATH, index=False)

print("Saved modeling data:", MODELING_PATH)
print("Shape:", model_df.shape)
print("Unique games:", model_df["game_id"].nunique())
print("Unique lineups:", model_df["lineup_key"].nunique())

print("\nSaved lineup summary:", LINEUP_SUMMARY_PATH)
print("Shape:", lineup_summary.shape)

## 13. Baseline recommender functions

These are simple functions your team can use before building machine learning models.

They recommend historically strong lineups by team and situation.

In [ ]:
def recommend_lineups(team_id: int, mode: str = "balanced", min_minutes: int = 50, top_n: int = 10) -> pd.DataFrame:
    candidates = lineup_summary[
        (lineup_summary["team_id"] == team_id)
        & (lineup_summary["total_minutes"] >= min_minutes)
    ].copy()

    if mode == "offense":
        sort_col = "shrunk_points_for_per_48"
        ascending = False
    elif mode == "defense":
        sort_col = "shrunk_points_against_per_48"
        ascending = True
    elif mode == "balanced":
        sort_col = "shrunk_net_points_per_48"
        ascending = False
    else:
        raise ValueError("mode must be one of: offense, defense, balanced")

    cols = [
        "team_id",
        "lineup_key",
        "total_minutes",
        "games",
        "points_for_per_48",
        "points_against_per_48",
        "net_points_per_48",
        "shrunk_points_for_per_48",
        "shrunk_points_against_per_48",
        "shrunk_net_points_per_48",
    ]

    return candidates.sort_values(sort_col, ascending=ascending)[cols].head(top_n)


# Example usage:
# recommend_lineups(team_id=1610612747, mode="offense", min_minutes=50, top_n=10)

## 14. Main EDA conclusions

1. The full-season team-perspective lineup dataset is usable.
2. Zero-duration rows should be filtered out before modeling.
3. Stint-level performance is noisy because most stints are short.
4. Lineup-level aggregation is more useful than individual stint rankings.
5. Minimum-minute thresholds matter a lot.
6. Raw per-48 ratings should be treated carefully.
7. Shrinkage-adjusted ratings are better for baseline recommendations.
8. Player-level summaries are useful, but they are not causal.
9. Context features like score margin, period, home/away, and opponent lineup should be included in modeling.

## 15. Recommended next EDA before modeling

Before training ML models, explore:

### A. Team-specific lineup rankings
Do the best lineups make basketball sense for each team?

### B. Stability by minutes threshold
Check whether top lineups stay similar at 20, 50, 100, and 200 minute thresholds.

### C. Context-specific rankings
Rank lineups separately for:
- Close games
- Trailing
- Leading
- Fourth quarter
- Overtime

### D. Opponent effects
Check whether certain lineups perform very differently depending on the opponent.

### E. Possession-based stats
Current data uses score-per-time stats. For a stronger model, add:
- possessions
- points per possession
- offensive rating
- defensive rating
- turnover rate
- free throw rate
- three-point attempt rate

### F. Modeling target decision
Decide whether the first model predicts:
- `points_for_per_48` for offense recommendations
- `net_points_per_48` for balanced recommendations
- future `offensive_rating` once possessions are added

### G. Train/test split strategy
Avoid random row splits only. Prefer splitting by date/game so the model is tested on future games.